# FINN integration test: TFC MNIST in our own DMA design

`fpga_top_test`: AXI DMA MM2S → **`finn_design_0`** (FINN stitched IP from `finn/tfc_example`) → AXI DMA S2MM.

One image per DMA transfer: 784 pixels (uint8, 196 words of 32 bit) go in, 1 byte comes back = the predicted digit.
The 4 test images from FINN's example must give the labels in `expected_output.npy` (7, 2, 1, 0).

Upload `finn_tfc_test.bit`, `finn_tfc_test.hwh`, `input.npy`, `expected_output.npy` next to this notebook.

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

S2MM_LENGTH = 0x58             # DMA register: bytes actually received by S2MM

ol = Overlay('finn_tfc_test.bit')
dma = ol.axi_dma_0
print('IP blocks :', list(ol.ip_dict.keys()))

images   = np.load('input.npy').astype(np.uint8).reshape(-1, 784)   # (4, 784), values 0..255
expected = np.load('expected_output.npy').reshape(-1)               # (4,)

tx = allocate(shape=(784,), dtype=np.uint8)
rx = allocate(shape=(1,),   dtype=np.uint8)

def classify(img):
    tx[:] = img
    tx.flush()
    rx[:] = 0xEE
    rx.flush()
    dma.recvchannel.transfer(rx)       # arm S2MM first
    dma.sendchannel.transfer(tx)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    rx.invalidate()
    return int(rx[0]), dma.read(S2MM_LENGTH)

## Classify the 4 test images

In [ ]:
ok = 0
for i, img in enumerate(images):
    label, n = classify(img)
    good = label == expected[i]
    ok += good
    print(f'image {i}: FPGA says {label}, expected {expected[i]}  {"ok" if good else "WRONG"}   (S2MM received {n} B)')
print('PASS' if ok == len(images) else f'FAIL ({len(images) - ok} wrong)')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
for i, a in enumerate(ax):
    a.imshow(images[i].reshape(28, 28), cmap='gray')
    a.set_title(f'FPGA: {classify(images[i])[0]}')
    a.axis('off')
plt.show()

## Speed (dominated by the Python DMA calls: the network itself needs ~8 µs per image)

In [ ]:
N = 1000
t0 = time.perf_counter()
for i in range(N):
    classify(images[i % len(images)])
t = time.perf_counter() - t0
print(f'{N} images in {t:.2f} s: {N/t:.0f} images/s, {t/N*1e6:.0f} µs per image')

In [ ]:
tx.freebuffer()
rx.freebuffer()